# Phase 3: Datenvorbereitung & Feature Engineering

**Ziel:** Bilder in ein Format bringen, das ML-Modelle verarbeiten können.

**StackFuel-Themen:** Native Python · Mathematik (Normalisierung, Vektoren)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from src.utils import get_class_dirs, load_images_as_array

print('Setup OK')

## 3.1 Daten laden

In [ ]:
IMG_SIZE = (64, 64)   # Größe reduzieren für schnelleres Training
MAX_PER_CLASS = 200   # Für Entwicklung begrenzen, später erhöhen

class_dirs = get_class_dirs('train')

X_list, y_list = [], []

for class_name, class_path in class_dirs.items():
    images = load_images_as_array(class_path, size=IMG_SIZE, max_images=MAX_PER_CLASS)
    X_list.append(images)
    y_list.extend([class_name] * len(images))
    print(f'{class_name}: {len(images)} Bilder geladen')

X = np.concatenate(X_list, axis=0)   # Shape: (N, 64, 64, 3)
y = np.array(y_list)

print(f'\nX Shape: {X.shape}')
print(f'y Shape: {y.shape}')
print(f'Pixelwerte: min={X.min():.2f}, max={X.max():.2f}  ← bereits normalisiert!')

## 3.2 Label Encoding

ML-Modelle brauchen Zahlen, keine Strings.

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Klassen-Mapping:')
for idx, cls in enumerate(le.classes_):
    print(f'  {idx} → {cls}')

## 3.3 Flatten: 3D-Bilder → 1D-Vektoren

Klassische ML-Modelle (Logistic Regression, Random Forest) erwarten 2D-Input: (n_samples, n_features).

In [ ]:
n_samples = X.shape[0]
X_flat = X.reshape(n_samples, -1)  # (N, 64*64*3) = (N, 12288)

print(f'Original Shape: {X.shape}')
print(f'Nach Flatten:   {X_flat.shape}')
print(f'Features pro Bild: {X_flat.shape[1]} (= 64 × 64 × 3 Pixel × Kanäle)')

## 3.4 Train/Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_flat, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded   # proportionale Klassenverteilung sicherstellen
)

print(f'Training:   {X_train.shape[0]} Bilder')
print(f'Validation: {X_val.shape[0]} Bilder')

# Klassenverteilung im Split prüfen
import pandas as pd
train_dist = pd.Series(le.inverse_transform(y_train)).value_counts(normalize=True).round(3)
print(f'\nKlassenverteilung Training:\n{train_dist.to_string()}')

## 3.5 Daten speichern (für Notebook 03)

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/classes.npy', le.classes_)

print('Daten gespeichert unter data/processed/')